In [4]:
!pip install -q -U langchain langchain-community langchain-google-genai

In [5]:
from google.colab import userdata
import os
os.environ["GOOGLE_API_KEY"]=userdata.get("GEMINI_API_KEYS")

MODEL="gemini-3.5-flash-lite"
EMB_MODEL="gemini-embedding-001"


In [11]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_google_genai import ChatGoogleGenerativeAI
@tool
def calculate_savings(income: int, expenditure: list) -> int:
  "This tool is to calculate saving of the person"
  cost=0
  for exp in expenditure:
    cost+=float(exp)
  return (income-cost)

@tool
def calculate_investement(P: float, T: float, R: float) -> float:
  "This is tool is calculate amount he will have if he investes P amount for T months period of time and for R percent rate of interest"
  SI=(P*T*R)/100
  return (P+SI)

@tool
def calcualate_goal_time(P: float, Price: float, Savings: float) -> float:
  "This tool is to calculate number of months required to pay the price where the user has P amount and saving (savings amount per month)"
  Price-=P
  return (Price/Savings)


llm=ChatGoogleGenerativeAI(model=MODEL)
llm_with_tools=llm.bind_tools([calcualate_goal_time,calculate_investement,calculate_savings])

def chat(query):
  messages=[]
  messages.append(HumanMessage(content=query))
  AI=llm_with_tools.invoke(messages)
  messages.append(AI)
  if not AI.tool_calls:
    print("INVALID RESPONSE")
    return
  tool_call=AI.tool_calls[0]
  tools={
      "calculate_savings":calculate_savings,
      "calculate_investement":calculate_investement,
      "calcualate_goal_time":calcualate_goal_time
  }
  selected_tool=tools[tool_call["name"]]
  tool_result=selected_tool.invoke(tool_call["args"])
  tool_message=ToolMessage(content=str(tool_result),name=tool_call["name"],tool_call_id=tool_call["id"])
  messages.append(tool_message)

  final_output=llm_with_tools.invoke(messages)
  return final_output.content[0]["text"]

print("Enter bye/exit/quit to terminate session")
while True:
  user = input("Enter your query: ").strip()
  if user.lower() in ['exit', 'bye', 'quit']:
    print("exited")
    break
  output = chat(user)
  print(output)

Enter bye/exit/quit to terminate session
Enter your query: I EARNED 100000 AND MY HOUSE RENT IS 10000 FOOD 10000 AND FUEL 4000
Based on your income and expenses, here is a breakdown of your finances:

* **Total Income:** 100,000
* **Total Expenses:** 24,000 (House Rent: 10,000 + Food: 10,000 + Fuel: 4,000)
* **Total Savings:** 76,000

You are saving 76,000 per month! Let me know if you would like to calculate investments or how long it will take to reach a specific financial goal with these savings.
Enter your query: i am investing 10000 for 10months at 10% 
If you invest 10,000 for 10 months at an interest rate of 10%, the total amount you will have is 20,000.
Enter your query: i want to buy thar car i have 500000 and its price is 2400000 and i save 50000per month
To buy the Thar (priced at ₹24,00,000), given that you currently have ₹5,00,000 and are saving ₹50,000 per month, it will take you **38 months** (or about 3 years and 2 months) to reach your goal.
Enter your query: bye
exite